# RFFMG (T5Chem) Molecular Generation Tutorial

This notebook extracts fragments from SMILES and generates molecules containing those
fragments using RFFMG (T5Chem). Run all generation cells with the `t5chem` kernel.

## Fragmentation Methods
- **BRICS**: Chemically meaningful fragmentation via BRICS decomposition.
- **RC_CMS**: Random-cut fragmentation (SP3-SP3 bonds, ring-ring connections, etc.).


## 0. Environment Setup

Run the following commands **in a terminal at the repository root** if the environment
is not already prepared. If it already exists, skip `conda create`.

```bash
conda create -n t5chem python=3.12.12 -y
conda run -n t5chem pip install -r requirements/t5chem_requirements.txt
conda run -n t5chem pip install -e .
conda run -n t5chem pip install ipykernel
conda run -n t5chem python -m ipykernel install --user --name t5chem --display-name "t5chem"
```

Select the **t5chem** kernel in Jupyter before running the cells below.
The first cell sets the working directory to the repository root so model, data,
and output paths work from this notebook's location in `tutorial/`.


In [ ]:
from pathlib import Path
import os

# Resolve paths consistently when opened from tutorial/ or the repository root.
working_dir = Path.cwd().resolve()
PROJECT_ROOT = next(
    (path for path in (working_dir, *working_dir.parents)
     if (path / 'setup.py').is_file() and (path / 'src' / 'func').is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Open this notebook from within the cloned repository.')
os.chdir(PROJECT_ROOT)
print(f'Project root: {PROJECT_ROOT}')


## 1. Download Models

Download pre-trained models from HuggingFace Hub (`sato-akinori/FFMG`).  
For private repositories, log in beforehand with `huggingface-cli login`.

The model paths in Section 2 follow this repository's layout
(`models/{repr_name}/{model_name}/{model_ver}/...`). If the downloaded archive expands
into a different layout, move the extracted directories so they match.

In [ ]:
import os
import glob
import subprocess
from huggingface_hub import snapshot_download

# Download models from HuggingFace
snapshot_download(
    repo_id='sato-akinori/FFMG',
    allow_patterns='models/*',
    local_dir='.'
)

# Extract zip files
for zip_file in glob.glob('models/**/*.zip', recursive=True):
    subprocess.run(['unzip', '-o', zip_file, '-d', os.path.dirname(zip_file)], check=True)
    os.remove(zip_file)


print('Model download complete.')


## 2. Configuration

Configure the fragmentation method, generation parameters, etc.

In [ ]:
# ========================================
# Configuration (modify as needed)
# ========================================
FRAG_METHOD  = 'rc_cms'      # 'brics' or 'rc_cms'
MODEL_VER    = 'finetuning'  # 'finetuning' or 'from_scratch'
SAMPLING_NUM = 5             # 5 or 10; from_scratch was trained on 5 only
N_SAMPLES    = 10            # Number of molecules to generate
NUM_BEAMS    = 10            # Number of beams for beam search
RANDOM_SEED  = 42            # Seed for rc_cms fragmentation; beam search itself is deterministic

# Model path relative to the repository root.
T5CHEM_MODEL_PATH = f'models/rffmg/t5chem/{MODEL_VER}/{FRAG_METHOD}/{SAMPLING_NUM}times_sampling/best_model/'


## 3. Import Libraries and Helper Functions

In [ ]:
import sys
import os
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

import pandas as pd
from rdkit import Chem
from rdkit.Chem import Draw
from IPython.display import display

from func.fragmentation import (
    BRICSFragmentize,
    RandomFragmentize,
    PostProcessSelectFrags,
)


def fragmentize_smiles(smiles, frag_method='brics', ratio=0.6, big_ring_thres=7, seed=42):
    """
    Fragmentize a SMILES string.

    Returns:
        pass_frags: Fragment SMILES (R-groups on rings trimmed)
    """
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        raise ValueError(f'Invalid SMILES: {smiles}')

    if frag_method == 'brics':
        frags = BRICSFragmentize(mol, returnSmiles=False)
        trim_r_on_ring = False
    elif frag_method == 'rc_cms':
        frags = RandomFragmentize(
            mol, returnSmiles=False,
            bigRingThres=big_ring_thres, rseed=seed, ratio=ratio
        )
        trim_r_on_ring = True
    else:
        raise ValueError(f'Unknown method: {frag_method}')

    if frags is None:
        return None

    pass_frags, _ = PostProcessSelectFrags(
        frags,
        smallCarbonFilter=True,
        trimRgroupOnRing=trim_r_on_ring,
        uniquenize=False,
    )
    return pass_frags


def draw_mols(smiles_list, legends=None, mols_per_row=4, img_size=(300, 300)):
    """Draw molecules from a list of SMILES strings."""
    mols = [Chem.MolFromSmiles(s) for s in smiles_list]
    mols = [m for m in mols if m is not None]
    if not mols:
        print('No valid molecules to draw.')
        return
    if legends is None:
        legends = [Chem.MolToSmiles(m) for m in mols]
    img = Draw.MolsToGridImage(
        mols[:12], molsPerRow=mols_per_row, subImgSize=img_size, legends=legends[:12]
    )
    display(img)


def generate_t5chem(
    fragments: str, model_path: str, num_beams: int, n_samples: int,
) -> list[str]:
    """Generate ranked candidates using T5Chem, retaining empty prediction slots.

    Args:
        fragments: Dot-separated fragment SMILES.
        model_path: T5Chem checkpoint directory.
        num_beams: Beam-search width.
        n_samples: Number of ranked candidates to return.

    Returns:
        Exactly n_samples SMILES strings in rank order, including empty strings.
    """
    import subprocess
    import tempfile

    if not 1 <= n_samples <= num_beams:
        raise ValueError('Require 1 <= n_samples <= num_beams.')

    with tempfile.TemporaryDirectory() as tmpdir:
        with open(os.path.join(tmpdir, 'test.source'), 'w') as f:
            f.write(fragments + '\n')
        with open(os.path.join(tmpdir, 'test.target'), 'w') as f:
            f.write('C\n')

        pred_file = os.path.join(tmpdir, 'predictions.csv')
        cmd = [
            't5chem', 'predict',
            '--data_dir', tmpdir,
            '--model_dir', model_path,
            '--prediction', pred_file,
            '--num_beams', str(num_beams),
            '--num_preds', str(n_samples),
            '--batch_size', '1',
        ]

        print(f'Running: {" ".join(cmd)}')
        subprocess.run(cmd, capture_output=True, text=True, check=True)

        df = pd.read_csv(pred_file, keep_default_na=False, dtype=str)
        prediction_columns = [f'prediction_{rank}' for rank in range(1, n_samples + 1)]
        missing_columns = [column for column in prediction_columns if column not in df.columns]
        if missing_columns:
            raise ValueError(f'Missing prediction columns: {missing_columns}')
        if len(df) != 1:
            raise ValueError(f'Expected one prediction row, got {len(df)}.')

        display(df)
        return df.loc[0, prediction_columns].tolist()


print('Library import complete.')


## 4. Input SMILES and Fragmentation

Enter an arbitrary SMILES and perform fragmentation.  
Modify `input_smiles` to try different molecules.

Repeated fragments are retained, matching `src/gen_frags/rffmg_frags.py`.
This example uses one cut pattern and all processed fragments. Dataset construction
also samples subsets over multiple patterns and applies a molecule/fragment size
filter. Use the same stored fragment set when comparing with a dataset result.


In [ ]:
# ========================================
# Input SMILES (modifiable)
# ========================================
input_smiles = 'CC(C)Cc1ccc(C(C)C(=O)O)cc1'  # Ibuprofen

# Validate and canonicalize SMILES
mol = Chem.MolFromSmiles(input_smiles)
assert mol is not None, 'Invalid SMILES. Please enter a valid SMILES string.'
canonical_smi = Chem.MolToSmiles(mol)
print(f'Input molecule (canonical SMILES): {canonical_smi}')

# Fragmentation
pass_frags = fragmentize_smiles(canonical_smi, frag_method=FRAG_METHOD, seed=RANDOM_SEED)

if pass_frags is None:
    print('This molecule cannot be fragmented. Try a different SMILES or method.')
else:
    print(f'\n--- Fragmentation Results ---')
    print(f'Method: {FRAG_METHOD}')
    print(f'Fragments: {pass_frags}')
    print(f'Number of fragments: {len(pass_frags.split("."))}')

    frag_smiles = pass_frags.split('.')
    draw_mols(
        [canonical_smi] + frag_smiles,
        legends=['Input molecule'] + [f'Fragment {i+1}' for i in range(len(frag_smiles))]
    )

---
## 5. Molecule Generation from Fragmented Input

Generate molecules from the fragments obtained in Section 4.

In [ ]:
assert pass_frags is not None, 'Fragmentation failed. Please check Section 4.'

print(f'Input fragments: {pass_frags}')
smiles = generate_t5chem(pass_frags, T5CHEM_MODEL_PATH, NUM_BEAMS, N_SAMPLES)
if smiles:
    draw_mols(smiles)

---
## 6. Generate Molecules from Custom Fragments

Instead of fragmenting a molecule first, you can directly specify fragment SMILES and generate molecules.

**Fragment format:**
- Use `[*]` or `*` to mark attachment points
- Separate multiple fragments with `.` (dot)
- Single fragment → scaffold decoration (fills attachment points)
- Multiple fragments → scaffold morphing (combines fragments into a molecule)

**Examples:**
- Single fragment: `c1ccc([*])cc1` (benzene with one attachment point)
- Multiple fragments: `[*]c1ccccc1.[*]C(=O)O` (benzene + carboxylic acid)

In [ ]:
# ========================================
# Input fragments directly (modifiable)
# ========================================
# Separate multiple fragments with '.'
input_fragments = ['OC1=C(O)C=CC([*])=C1.[*]N[*]', 'OC1=C(O)C=CC([*])=C1', 'OC1=C(C=CC=C1)O.[*]N[*]', 'OC1=C(C=CC=C1)O']
input_fragments = [[Chem.MolToSmiles(Chem.MolFromSmiles(frag)) for frag in input_fragment.split('.')] for input_fragment in input_fragments]

custom_frags = ['.'.join(input_fragment) for input_fragment in input_fragments]
print(f'Input fragments (canonical): {custom_frags}')
print(f'Number of fragments: {len([frag for frag_list in input_fragments for frag in frag_list])}')

for custom_frag in custom_frags:
    draw_mols(custom_frag.split('.'), legends=[f'Fragment {i+1}' for i in range(len(custom_frag.split('.')))])

### Generate from Custom Fragments


In [ ]:
results = list()
for custom_frag in custom_frags:
    print(f'Input fragments: {custom_frag}')
    smiles = generate_t5chem(custom_frag, T5CHEM_MODEL_PATH, NUM_BEAMS, N_SAMPLES)
    if smiles:
        draw_mols(smiles)
    results.append([custom_frag] + smiles)
    
# Keep one column per requested rank, including empty prediction slots.
gen_smiles_df = pd.DataFrame(results, columns=['input'] + [f'top-{k}' for k in range(1, N_SAMPLES + 1)])
output_dir = f'results/rffmg/t5chem/{MODEL_VER}/{FRAG_METHOD}/{SAMPLING_NUM}times_sampling/beam/custom_frag/'
os.makedirs(output_dir, exist_ok=True)
gen_smiles_df.to_csv(f'{output_dir}/gen_smiles.csv', index=False)